# 🏭 Delhi Deep Dive — Module 04: Air Quality Risk
## PM2.5 pollution scoring for 11 Delhi districts (2022–2023 real data + historical estimates)

In [ ]:
import os, time, warnings, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../data/outputs/delhi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

AQ_URL    = 'https://air-quality-api.open-meteo.com/v1/air-quality'
START_AQ  = '2022-01-01'
END_AQ    = '2023-12-31'

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}

# Historical PM2.5 scaling (CPCB trend data)
HISTORICAL_SCALE = {
    2010: 1.80, 2011: 1.75, 2012: 1.70, 2013: 1.65, 2014: 1.60,
    2015: 1.55, 2016: 1.45, 2017: 1.35, 2018: 1.25, 2019: 1.15,
    2020: 0.90,  # COVID lockdown
    2021: 1.10, 2022: 1.00, 2023: 0.95,
}

def score_to_category(s):
    if s <= 25: return 'LOW'
    elif s <= 50: return 'MEDIUM'
    elif s <= 75: return 'HIGH'
    return 'VERY HIGH'

def fetch_with_retry(url, params, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            wait = 5 * (2 ** attempt)
            print(f'⏳ {wait}s…', end=' ', flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError('Max retries')

print('Air Quality module setup complete.')
print('Note: Real AQ data available 2022–2023. 2010–2021 uses calibrated historical estimates.')

## Fetch Real PM2.5 Data (2022–2023)

In [ ]:
aq_dfs = []
for district, coords in DELHI_DISTRICTS.items():
    print(f'  {district}…', end=' ', flush=True)
    try:
        params = {
            'latitude': coords['lat'], 'longitude': coords['lon'],
            'hourly': ['pm2_5', 'pm10', 'european_aqi'],
            'start_date': START_AQ, 'end_date': END_AQ,
        }
        data = fetch_with_retry(AQ_URL, params)['hourly']
        df = pd.DataFrame({
            'datetime': pd.to_datetime(data['time']),
            'pm2_5':    pd.to_numeric(data['pm2_5'],        errors='coerce'),
            'pm10':     pd.to_numeric(data['pm10'],         errors='coerce'),
            'aqi':      pd.to_numeric(data['european_aqi'], errors='coerce'),
        })
        df['date'] = df['datetime'].dt.date
        daily = df.groupby('date').agg(
            pm25_max=('pm2_5','max'), pm25_mean=('pm2_5','mean'),
            pm10_mean=('pm10','mean'), aqi_max=('aqi','max')
        ).reset_index()
        daily['date']    = pd.to_datetime(daily['date'])
        daily['district']= district
        daily['lat']     = coords['lat']
        daily['lon']     = coords['lon']
        aq_dfs.append(daily)
        print('✓')
    except Exception as e:
        print(f'⚠({e})')
    time.sleep(1.5)

if aq_dfs:
    daily_all = pd.concat(aq_dfs, ignore_index=True)
    daily_all['year']  = daily_all['date'].dt.year
    daily_all['month'] = daily_all['date'].dt.month
    print(f'\nFetched {len(daily_all):,} daily AQ records across {daily_all["district"].nunique()} districts')
    print(f'Annual avg PM2.5: {daily_all["pm25_mean"].mean():.1f} µg/m³')
else:
    print('⚠ No AQ data fetched')
    daily_all = pd.DataFrame()

## Build Full 2010–2023 Dataset (Historical Scaling)

In [ ]:
# Get baseline PM2.5 per district from real data
if len(daily_all) > 0:
    base_pm25 = daily_all.groupby('district')['pm25_mean'].mean().to_dict()
else:
    base_pm25 = {d: 80.0 for d in DELHI_DISTRICTS}

all_rows = []
for district in DELHI_DISTRICTS:
    base = base_pm25.get(district, 80.0)
    if np.isnan(base): base = 80.0
    for year in range(2010, 2024):
        scale   = HISTORICAL_SCALE.get(year, 1.0)
        pm25_yr = base * scale
        all_rows.append({
            'district':            district,
            'year':                year,
            'lat':                 DELHI_DISTRICTS[district]['lat'],
            'lon':                 DELHI_DISTRICTS[district]['lon'],
            'avg_pm25_year':       round(pm25_yr, 1),
            'days_pm25_above_60':  min(int(pm25_yr * 2.2), 365),
            'days_pm25_above_150': min(int(pm25_yr * 0.55), 180),
            'max_aqi_year':        round(pm25_yr * 4.5, 1),
            'bad_air_months':      min(int(pm25_yr * 0.08), 12),
            'winter_pm25_avg':     round(pm25_yr * 1.6, 1),
            'airquality_label':    int(pm25_yr * 2.2 > 120),
        })

yearly = pd.DataFrame(all_rows)
print(f'Full dataset: {yearly.shape}')
print(f"Label distribution: {yearly['airquality_label'].value_counts().to_dict()}")

## Train XGBoost + Score + Save

In [ ]:
FEATURES = ['avg_pm25_year','days_pm25_above_60','days_pm25_above_150',
            'max_aqi_year','bad_air_months','winter_pm25_avg']
TARGET   = 'airquality_label'

train = yearly[yearly['year']<=2021].dropna(subset=FEATURES+[TARGET])
test  = yearly[yearly['year']>=2022].dropna(subset=FEATURES+[TARGET])

model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
model.fit(train[FEATURES].astype(float), train[TARGET].astype(int))
if len(test) > 0:
    print(f'Test acc: {accuracy_score(test[TARGET], model.predict(test[FEATURES].astype(float))):.3f}')

full  = yearly.dropna(subset=FEATURES+[TARGET]).copy()
probs = model.predict_proba(full[FEATURES].astype(float))[:,1]
full['airquality_prob']       = probs.round(4)
full['airquality_risk_score'] = (probs * 100).round(2)
full['risk_category']         = full['airquality_risk_score'].apply(score_to_category)

# Feature importance plot
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_}).sort_values('importance')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat_df['feature'], feat_df['importance'], color='purple', alpha=0.8)
ax.set_title('Delhi Air Quality — Feature Importances')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_airquality_feature_importance.png', dpi=150)
plt.show()

result = full[['district','year','lat','lon','airquality_label','airquality_prob','airquality_risk_score','risk_category']]
result.to_csv(f'{OUTPUT_DIR}/delhi_airquality_scores.csv', index=False)
with open(f'{OUTPUT_DIR}/delhi_airquality_model.pkl','wb') as f:
    pickle.dump(model, f)
print(f'Saved delhi_airquality_scores.csv ({len(result)} rows)')
print(result[result['year']==2023][['district','airquality_risk_score','risk_category']].to_string())